In [139]:
import requests
from bs4 import BeautifulSoup
import json
from tqdm import tqdm

In [7]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36'
}

cookie_string = 'Hm_lvt_fa659f678fedf5ea21219e8fb2914831=1718603157; Hm_lpvt_fa659f678fedf5ea21219e8fb2914831=1718603157; sgst-instr=0ED024737D6AF1C34606A6E119EB8D63; Hm_lvt_297c9349622a15b64a77366360801f57=1718603232; Hm_lpvt_297c9349622a15b64a77366360801f57=1718603232'

# 将cookie字符串转换为字典
cookies_dict = {cookie.split('=')[0]: cookie.split('=')[1] for cookie in cookie_string.split('; ')}


{'Hm_lvt_fa659f678fedf5ea21219e8fb2914831': '1718603157', 'Hm_lpvt_fa659f678fedf5ea21219e8fb2914831': '1718603157', 'sgst-instr': '0ED024737D6AF1C34606A6E119EB8D63', 'Hm_lvt_297c9349622a15b64a77366360801f57': '1718603232', 'Hm_lpvt_297c9349622a15b64a77366360801f57': '1718603232'}


In [151]:

response = requests.post("https://cs1.sgst.cn/recordBase/listPage/list", headers=headers, cookies=cookies_dict,json={"page":1,"limit":500,"serviceRecordCode":"","nameCh":"","appCompany":"","subDwAccount":"","finishDate":"","finishDateStart":"","finishDateEnd":"","appDateStart":"","appDateEnd":"","invcCode":"","invcNum":"","deptName":"","completeFlag":"0","recordType":"仪器"})
# response.json()

# deviceInfo=[{
#     "recordAppCompany":"中国科学院上海高等研究院",
#     "recordId":"f52ba2f5-78ac-4199-a9a2-66cabe3a7937",
# }]

deviceInfo=[]

for device in response.json()["data"]["content"]:
    # print(device["recordAppCompany"])
    # print(device["recordId"])

    deviceInfo.append({
        "recordAppCompany":device["recordAppCompany"],
        "recordId":device["recordId"]
    })



JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [150]:
for device in tqdm(deviceInfo):
    url = f'https://cs1.sgst.cn/recordIns/updatePage/{device["recordId"]}'  # 替换为目标URL
    response = requests.get(url, headers=headers, cookies=cookies_dict)
    soup = BeautifulSoup(response.text, 'html.parser')

    # 辅助函数：获取输入字段的值，如果不存在则返回空字符串
    def get_input_value(soup, input_id):
        input_element = soup.find('input', {'id': input_id})
        if input_element is None:
            input_element = soup.find('input', {'name': input_id})
            if input_element.attrs["type"]=="radio":
                return [radio['value'] for radio in soup.find_all('input', {'name': input_id, 'type': 'radio'}) if radio.has_attr('checked')][0]

            return input_element.attrs["value"]
        if not input_element.has_attr("value"):
            return soup.find('select', {'id': input_id}).find('option', selected=True)['value'],
        return input_element['value']
        # return input_element['value'] if input_element and input_element.has_attr('value') else ""

    # 提取数据
    data = {
        "serviceRecord": {
            "source": soup.find('input', {'name': "source"}).attrs["value"],
            "getInvoiceFlag": get_input_value(soup, 'getInvoiceFlag'),
            "attachPath": get_input_value(soup, 'invcImageUrl'),
            "imageId": get_input_value(soup, 'invoiceImage'),
            "file": "",  # 没有找到相关信息，保留为空
            "isImg": "false",  # 根据上下文推断此值
            "invcCodeOcr": "",  # 没有找到相关信息，保留为空
            "invcNoOcr": get_input_value(soup, 'invcNoOcr'),
            "amountOcr": get_input_value(soup, 'amountOcr'),
            "invcDateOcr": get_input_value(soup, 'invcDateOcr'),
            "sellerName": get_input_value(soup, 'sellerName'),
            "purchaserName": get_input_value(soup, 'purchaserName'),
            "invcCode": get_input_value(soup, 'invcCode'),
            "invcNo": get_input_value(soup, 'invcNo'),
            "amount": get_input_value(soup, 'amount'),
            "invcDate": get_input_value(soup, 'invcDate'),
            "appCompName": get_input_value(soup, 'appCompName'),
            "delegtId": "",
            "dwType": "研究院所",  # 根据上下文推断此值
            "appAddress": get_input_value(soup, 'appAddress'),
            "appUserName": get_input_value(soup, 'appUserName'),
            "appPhone": get_input_value(soup, 'appPhone'),
            "province": "上海市",
            "city": "浦东新区",
            "appArea": "上海市-浦东新区",  # 根据上下文推断此值
            "trade": soup.find('select', {'id': 'trade'}).find('option', selected=True)['value'],
            "appPostcode": "",  # 没有找到相关信息，保留为空
            "appEmail": "",  # 没有找到相关信息，保留为空
            "appFax": "",  # 没有找到相关信息，保留为空
            "appBankName": "",  # 没有找到相关信息，保留为空
            "appBankAccount": "",  # 没有找到相关信息，保留为空
            "serviceCount": get_input_value(soup, 'serviceCount'),
            "operatorName": soup.find('select', {'id': 'operatorName'}).find('option', selected=True)['value'],
            "appCount": get_input_value(soup, 'appCount'),
            "appHour": get_input_value(soup, 'appHour'),
            "appFee": get_input_value(soup, 'appFee'),
            "proName": "",  # 没有找到相关信息，保留为空
            "proCode": "",  # 没有找到相关信息，保留为空
            "serviceType": soup.find('select', {'id': 'serviceType'}).find('option', selected=True)['value'],
            "select": "电子与通信技术",  # 根据上下文推断此值
            "subjectArea": "电子与通信技术",  # 根据上下文推断此值
            "isAgreement": get_input_value(soup, 'isAgreement'),
            "outUseAddress": "",  # 没有找到相关信息，保留为空
            "customsNum": "",  # 没有找到相关信息，保留为空
            "proFrom": "",  # 没有找到相关信息，保留为空
            "estimate": get_input_value(soup, 'estimate'),
            "appRemk": "",  # 没有找到相关信息，保留为空
            "recordType": get_input_value(soup, 'recordType'),
            "id": get_input_value(soup, 'id'),
            "invcId": get_input_value(soup, 'invcId'),
            "macId": get_input_value(soup, 'macId'),
            "invcDeptName": device["recordAppCompany"]
        }
    }

    data["invoice"]=data["serviceRecord"]

    # print(data)
    # break
    res=requests.post(url="https://cs1.sgst.cn/recordIns/updatePage/submit",cookies=cookies_dict,json=data)

    # 使用 json.loads 将响应文本解析为 Python 字典
    response_json = json.loads(res.text)

    # 将 Unicode 转换为可读的中文字符
    # print(device)
    # print(response_json['message'])

    if response_json['message']!="提交成功":
        print(response_json['message'])


{'recordAppCompany': '张江国家实验室', 'recordId': '8df3b238-2310-4525-ae27-dd5779553f2b'}
提交成功
{'recordAppCompany': '张江国家实验室', 'recordId': '8cee72a3-357f-4ee3-b7b3-865cb7f9dd99'}
提交成功
{'recordAppCompany': '张江国家实验室', 'recordId': '88c1c0f7-f414-4d3c-a131-e72c17084a1d'}
提交成功
{'recordAppCompany': '张江国家实验室', 'recordId': '8857c480-ff66-4243-8464-a5562133b86f'}
提交成功
{'recordAppCompany': '张江国家实验室', 'recordId': '8538eddc-6bfd-4169-a62d-fe6660dbffbb'}
提交成功
{'recordAppCompany': '张江国家实验室', 'recordId': '82f892b6-0789-4e78-b6bf-170fc3fb5ca5'}
提交成功
{'recordAppCompany': '张江国家实验室', 'recordId': '828700ae-4a2a-47fa-a2ac-f29edf0db918'}
提交成功
{'recordAppCompany': '张江国家实验室', 'recordId': '802a3b1a-b3d6-4b5a-a189-0494e74dce3b'}
提交成功
{'recordAppCompany': '张江国家实验室', 'recordId': '7fb2212e-e54c-4c35-b5c0-65d5add5e685'}
提交成功
{'recordAppCompany': '张江国家实验室', 'recordId': '748771f1-483f-4432-a71a-8672288ef39d'}
提交成功
{'recordAppCompany': '张江国家实验室', 'recordId': '73ad8514-0444-419b-8b0c-cb035e6aa39a'}
提交成功
{'recordAppCompany': 

In [143]:
correct={"serviceRecord":{"source":"2","getInvoiceFlag":"0","attachPath":"8f7916e7-c535-45c5-8528-a63c26512591.pdf","imageId":"4d14d785-41aa-4006-9c30-61c30f8c2d84","file":"","isImg":"false","invcCodeOcr":"","invcNoOcr":"24312000000119764787","amountOcr":"9521.49","invcDateOcr":"2024-04-28","sellerName":"上海科技大学","purchaserName":"中国科学院上海高等研究院","invcCode":"","invcNo":"24312000000119764787","amount":"9521.49","invcDate":"2024-04-28","appCompName":"中国科学院上海高等研究院","delegtId":"","dwType":"研究院所","appAddress":"上海市浦东新区张江高科技园区海科路99号","appUserName":"冯涛","appPhone":"021-20325085","province":"上海市","city":"浦东新区","appArea":"上海市-浦东新区","trade":"电子/信息技术","appPostcode":"","appEmail":"","appFax":"","appBankName":"","appBankAccount":"","serviceCount":"1","operatorName":"杨津津","appCount":"1","appHour":"0.00","appFee":"91.01","proName":"","proCode":"","serviceType":"委托共享","select":"电子与通信技术","subjectArea":"电子与通信技术","isAgreement":"0","outUseAddress":"","customsNum":"","proFrom":"","estimate":"好","appRemk":"","recordType":"仪器","id":"ecb52004-69ca-4c24-bb4e-737f90c28cd9","invcId":"39d0bc03-ca25-45f9-9582-46a35274c3e2","macId":"239afac0-98db-4a11-b813-45c31013ecf1","invcDeptName":"中国科学院上海高等研究院"},"invoice":{"source":"2","getInvoiceFlag":"0","attachPath":"8f7916e7-c535-45c5-8528-a63c26512591.pdf","imageId":"4d14d785-41aa-4006-9c30-61c30f8c2d84","file":"","isImg":"false","invcCodeOcr":"","invcNoOcr":"24312000000119764787","amountOcr":"9521.49","invcDateOcr":"2024-04-28","sellerName":"上海科技大学","purchaserName":"中国科学院上海高等研究院","invcCode":"","invcNo":"24312000000119764787","amount":"9521.49","invcDate":"2024-04-28","appCompName":"中国科学院上海高等研究院","delegtId":"","dwType":"研究院所","appAddress":"上海市浦东新区张江高科技园区海科路99号","appUserName":"冯涛","appPhone":"021-20325085","province":"上海市","city":"浦东新区","appArea":"上海市-浦东新区","trade":"电子/信息技术","appPostcode":"","appEmail":"","appFax":"","appBankName":"","appBankAccount":"","serviceCount":"1","operatorName":"杨津津","appCount":"1","appHour":"0.00","appFee":"91.01","proName":"","proCode":"","serviceType":"委托共享","select":"电子与通信技术","subjectArea":"电子与通信技术","isAgreement":"0","outUseAddress":"","customsNum":"","proFrom":"","estimate":"好","appRemk":"","recordType":"仪器","id":"ecb52004-69ca-4c24-bb4e-737f90c28cd9","invcId":"39d0bc03-ca25-45f9-9582-46a35274c3e2","macId":"239afac0-98db-4a11-b813-45c31013ecf1","invcDeptName":"中国科学院上海高等研究院"}}

def compare_json(json1, json2, path=""):
    differences = {}

    for key in json1.keys():
        if key not in json2:
            differences[path + key] = (json1[key], "Missing in json2")
        else:
            if isinstance(json1[key], dict) and isinstance(json2[key], dict):
                differences.update(compare_json(json1[key], json2[key], path + key + "."))
            elif json1[key] != json2[key]:
                differences[path + key] = (json1[key], json2[key])

    for key in json2.keys():
        if key not in json1:
            differences[path + key] = ("Missing in json1", json2[key])

    return differences

differences = compare_json(correct, data)

# 打印出所有不同的项目
for path, value in differences.items():
    print(f"Difference at {path}: {value[0]} != {value[1]}")


In [135]:
response

<Response [200]>

In [121]:
res=requests.post(url="https://cs1.sgst.cn/recordIns/updatePage/submit",cookies=cookies_dict,json=data)

# Selenium

In [152]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 初始化 Chrome 浏览器
driver = webdriver.Chrome()


In [155]:
# 添加 Cookie
cookie = {
    'name': 'your_cookie_name',
    'value': 'your_cookie_value',
    'domain': 'baidu.com'
}
driver.add_cookie(cookie)

driver.get('https://bilibili.com/')

In [ ]:
# 提交表单
submit_button = driver.find_element(By.CSS_SELECTOR, "a[lay-submit][lay-filter='submit']")
driver.execute_script("arguments[0].click();", submit_button)

In [156]:

# 等待响应并获取结果

# 关闭浏览器
driver.quit()